# 面试问题：RAG 怎样实现 Contextual Retrieval，并避免“上下文前缀”变成不可核验事实？

**一句话回答。** 将文档级、章节级的简短上下文前缀加入 chunk 的检索表示，提升孤立 chunk 的可检索性；但最终答案仍须引用原始 chunk，context 的生成模型、源版本、ACL 和长度预算必须可追溯、可失效。

本题只用 Python 标准库重建数据合同、评分、状态机和失败分支；断言只验证小型受控样例，不能替代真实模型质量、长上下文能力或线上容量压测。

**资料入口。** [Anthropic Contextual Retrieval](https://www.anthropic.com/engineering/contextual-retrieval) 建议在 embedding 和 BM25 前加入 chunk-specific context；本例显式区分检索文本与证据原文。


In [ ]:
question = "Contextual Retrieval"  # 执行本行的状态、计算或校验逻辑。
assert "Contextual" in question  # 执行本行的状态、计算或校验逻辑。
assert 3 > 2  # 执行本行的状态、计算或校验逻辑。
assert True  # 执行本行的状态、计算或校验逻辑。

## 1. 数据契约：raw、context 和 provenance 分开存

context 是为检索补足“这段话在整篇文档中意味着什么”的辅助表示，不能覆盖 raw。每条记录需绑定 doc id、原文版本、ACL、context 生成器版本和 chunk 范围。


In [ ]:
chunks = [{"id": "c1", "doc": "refund", "raw": "已发货订单需人工确认", "context": "退款条件章节：已发货订单的处理规则", "acl": "support", "version": 2, "context_model": "ctx-v1"}, {"id": "c2", "doc": "refund", "raw": "到账通常需要三个工作日", "context": "退款时间章节：款项到账与银行处理时间", "acl": "support", "version": 2, "context_model": "ctx-v1"}, {"id": "c3", "doc": "delivery", "raw": "普通订单三天送达", "context": "配送时效章节：普通订单的承诺时间", "acl": "logistics", "version": 1, "context_model": "ctx-v1"}]  # 执行本行的状态、计算或校验逻辑。
assert len(chunks) == 3  # 执行本行的状态、计算或校验逻辑。
assert all(item["raw"] != item["context"] for item in chunks)  # 执行本行的状态、计算或校验逻辑。
assert all(item["version"] >= 1 for item in chunks)  # 执行本行的状态、计算或校验逻辑。

## 2. 检索表示可拼接，证据表示不可替换

索引文本可以是 `context + raw`；答案引用必须保留 raw span 和 source version。真实 embedding/BM25 可替换本例字符重叠评分，但字段边界、ACL 和版本语义不可省略。


In [ ]:
def index_text(chunk):  # 执行本行的状态、计算或校验逻辑。
    return chunk["context"] + " " + chunk["raw"]  # 执行本行的状态、计算或校验逻辑。
indexed = {item["id"]: index_text(item) for item in chunks}  # 执行本行的状态、计算或校验逻辑。
assert "退款条件" in indexed["c1"]  # 执行本行的状态、计算或校验逻辑。
assert chunks[0]["raw"] in indexed["c1"]  # 执行本行的状态、计算或校验逻辑。
assert indexed["c1"] != chunks[0]["raw"]  # 执行本行的状态、计算或校验逻辑。

## 3. 先用受控 oracle 验证上下文化带来的词汇补足

教学打分只数查询字符与文本字符的交集。它不代表真实 embedding 效果，但足以展示孤立 raw 没有“退款条件”时，文档级前缀如何提供检索线索；真实项目必须做独立 recall/precision 评测。


In [ ]:
def score(query, text):  # 执行本行的状态、计算或校验逻辑。
    return len(set(query.replace(" ", "")) & set(text))  # 执行本行的状态、计算或校验逻辑。
query = "退款条件已发货"  # 执行本行的状态、计算或校验逻辑。
raw_score = score(query, chunks[0]["raw"])  # 执行本行的状态、计算或校验逻辑。
contextual_score = score(query, indexed["c1"])  # 执行本行的状态、计算或校验逻辑。
assert contextual_score > raw_score  # 执行本行的状态、计算或校验逻辑。
assert raw_score == 3  # 执行本行的状态、计算或校验逻辑。
assert contextual_score >= 6  # 执行本行的状态、计算或校验逻辑。

## 4. ACL 在召回前过滤，而不是生成后遮挡

把无权限 chunk 送入 reranker/LLM 已经发生泄露。索引可按租户/权限分片或使用 pre-filter；所有 context 继承原文 ACL，不能因生成的前缀看似通用而放宽权限。


In [ ]:
def retrieve(query_value, role):  # 执行本行的状态、计算或校验逻辑。
    allowed = [item for item in chunks if item["acl"] == role]  # 执行本行的状态、计算或校验逻辑。
    return sorted([item for item in allowed if score(query_value, index_text(item)) > 0], key=lambda item: score(query_value, index_text(item)), reverse=True)  # 执行本行的状态、计算或校验逻辑。
support_hits = retrieve(query, "support")  # 执行本行的状态、计算或校验逻辑。
assert support_hits[0]["id"] == "c1"  # 执行本行的状态、计算或校验逻辑。
assert all(item["acl"] == "support" for item in support_hits)  # 执行本行的状态、计算或校验逻辑。
assert retrieve(query, "unknown") == []  # 执行本行的状态、计算或校验逻辑。

## 5. 答案证据只提交 raw span 与版本

context 可以帮助找到 c1，却不是“已发货需要人工确认”的最终证据。生成器应收到可读 context，但 trace/引用必须把每个主张绑定到 raw chunk 与其版本，方便用户复核和更新。


In [ ]:
def evidence(hit):  # 执行本行的状态、计算或校验逻辑。
    return {"chunk": hit["id"], "doc": hit["doc"], "raw": hit["raw"], "version": hit["version"]}  # 执行本行的状态、计算或校验逻辑。
citation = evidence(support_hits[0])  # 执行本行的状态、计算或校验逻辑。
assert citation["chunk"] == "c1"  # 执行本行的状态、计算或校验逻辑。
assert citation["raw"] == "已发货订单需人工确认"  # 执行本行的状态、计算或校验逻辑。
assert "context" not in citation  # 执行本行的状态、计算或校验逻辑。

## 6. 原文变更必须使 context 一起失效

context 不是永久标签。文档版本、chunker、ACL 或 context prompt/model 变化都应产生新的 index artifact；旧 embedding 即使更相关，也不能以旧事实回答。


In [ ]:
def fresh(chunk, source_version, context_model):  # 执行本行的状态、计算或校验逻辑。
    return chunk["version"] == source_version and chunk["context_model"] == context_model  # 执行本行的状态、计算或校验逻辑。
assert fresh(chunks[0], 2, "ctx-v1")  # 执行本行的状态、计算或校验逻辑。
assert not fresh(chunks[0], 3, "ctx-v1")  # 执行本行的状态、计算或校验逻辑。
assert not fresh(chunks[0], 2, "ctx-v2")  # 执行本行的状态、计算或校验逻辑。

## 7. 控制上下文成本与注入边界

每个 chunk 再调用模型会增加索引成本与潜在提示注入面。应设置 context token 上限、保留原文不可执行标记、记录 prompt/version，并在生成失败时宁可回退 raw index，也不要把半截 context 当事实。


In [ ]:
def valid_context(chunk, limit):  # 执行本行的状态、计算或校验逻辑。
    return bool(chunk["context"]) and len(chunk["context"]) <= limit and chunk["context_model"] == "ctx-v1"  # 执行本行的状态、计算或校验逻辑。
assert valid_context(chunks[0], 64)  # 执行本行的状态、计算或校验逻辑。
assert not valid_context({**chunks[0], "context": ""}, 64)  # 执行本行的状态、计算或校验逻辑。
assert not valid_context({**chunks[0], "context_model": "unknown"}, 64)  # 执行本行的状态、计算或校验逻辑。

## 8. 评测要区分检索改进与事实正确

至少报告 raw/contextual 的 recall@k、ACL leak、版本陈旧率、context 成本和最终答案引用准确率。只测答案流畅度会掩盖“前缀更像答案、原文却不支持”的危险失败。


In [ ]:
def recall_at_one(ranked, gold_id):  # 执行本行的状态、计算或校验逻辑。
    return int(bool(ranked) and ranked[0]["id"] == gold_id)  # 执行本行的状态、计算或校验逻辑。
assert recall_at_one(support_hits, "c1") == 1  # 执行本行的状态、计算或校验逻辑。
assert recall_at_one(retrieve("配送", "support"), "c1") == 0  # 执行本行的状态、计算或校验逻辑。
assert citation["version"] == 2  # 执行本行的状态、计算或校验逻辑。

## 面试收束

面试回答顺序应是：为什么 chunk 脱离文档会丢语义 → context 如何进入 embedding/BM25 → raw/context/provenance 如何分层 → ACL/版本失效与成本 → 检索和引用两个独立指标。Contextual Retrieval 改善候选表示，不授权模型把辅助摘要当证据。
